# Lab 04.2 — Logitech C270 USB Webcam

## Objectives

- verify that Linux detects the webcam;
- open the camera with OpenCV;
- capture a single frame;
- display camera images correctly inside Jupyter.

Connect the **Logitech C270** to the Zybo USB host port before starting.

## 1. Check the Linux video device

The following commands help identify the USB camera and its `/dev/video*` device.

In [ ]:
!lsusb
!ls -l /dev/video* 2>/dev/null || echo "No /dev/video device found"

A webcam may expose more than one `/dev/videoX` node. We start with device 0. If it cannot provide frames, try 1.

In [ ]:
import cv2
import matplotlib.pyplot as plt

CAMERA_INDEX = 0

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Camera opened:", cap.isOpened())

## 2. Capture one frame

In [ ]:
ret, frame = cap.read()

print("Frame captured:", ret)

if ret:
    print("Frame shape:", frame.shape)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(frame_rgb)
    plt.title("Logitech C270 frame")
    plt.axis("off");

## 3. Capture several frames inline

Jupyter on the board does not require a desktop GUI, therefore we avoid `cv2.imshow()`.

In [ ]:
from IPython.display import display, Image
import time

for i in range(20):
    ret, frame = cap.read()
    if not ret:
        print("Frame capture failed.")
        break

    ok, encoded = cv2.imencode(".jpg", frame)
    if ok:
        display(Image(data=encoded.tobytes()))
        
    time.sleep(0.05)

    # Keep the notebook output compact.
    if i < 19:
        from IPython.display import clear_output
        clear_output(wait=True)

## 4. Release the camera

This step is important. A camera left open by one notebook may not be available to the next notebook.

In [ ]:
cap.release()
print("Camera released.")

## Troubleshooting

If `Camera opened` is `False` or `ret` is `False`:

1. check `/dev/video*`;
2. try `CAMERA_INDEX = 1`;
3. disconnect and reconnect the webcam;
4. release any camera object opened by another notebook;
5. keep the first tests at 640×480.

## Exercise

Move an object in front of the camera and capture a frame in which the object is clearly visible. Record the image resolution reported by OpenCV.